# 4-D phase space reconstruction with `ResNNTransform`
This is a copy of [`example_reconstruction_4d.ipynb`](example_reconstruction_4d.ipynb) that swaps in `gpsr.beams.ResNNTransform` -- a residual-network beam transformer that first applies a linear transformation to the base distribution and then adds a learned nonlinear residual -- in place of the default `NNTransform`. It also compares the linear (first-layer) prediction and the final reconstructed beam distribution to the ground-truth distribution.

### Python package imports

In [ ]:
import numpy as np
import torch
import lightning as L
from cheetah.particles import ParticleBeam
from gpsr.modeling import GPSR, GPSRQuadScanLattice
from gpsr.train import train_gpsr_multistep
from gpsr.beams import NNParticleBeamGenerator, ResNNTransform
from gpsr.datasets import QuadScanDataset, split_dataset
from gpsr.utils import to_linear_beam

### Load data

Load measurement dataset and split into train and test datasets

In [ ]:
dset = torch.load(
    "example_data/example_datasets/reconstruction_4D.dset", weights_only=False
)
print(
    dset.parameters.shape,
    dset.observations[0].shape,
    dset.screen,
)
dset.plot_data();

In [ ]:
train_k_ids = np.arange(0, len(dset.parameters), 2)
train_dset, test_dset = split_dataset(dset, train_k_ids)

In [ ]:
train_dset.plot_data();

In [ ]:
test_dset.plot_data();

### Create the quadrupole scan lattice
Here we use the differentiable Cheetah `Screen`. This screen uses kernel desity estimation to approximate the histogram in order to make it differentiable and vectorized.

In [ ]:
# print screen information
print(train_dset.screen)
# create diagnostic lattice
p0c = 43.36e6  # reference momentum in eV/c
gpsr_lattice = GPSRQuadScanLattice(l_quad=0.1, l_drift=1.0, screen=train_dset.screen)

### Define the GPSR model for training
The GPSR model contains the ML-based parameterization of the initial beam distribution `NNParticleBeamGenerator` with 10k particles, using `ResNNTransform` (with a skip connection) as its transformer, and the differentiable simulation of the diagnostic lattice (same one used above to generate the training data).

In [ ]:
transformer = ResNNTransform(
    n_hidden=2, width=20, phase_space_dim=4, use_skip_connection=True
)
gpsr_model = GPSR(
    NNParticleBeamGenerator(10000, p0c, transformer=transformer, n_dim=4), gpsr_lattice
)
train_loader = torch.utils.data.DataLoader(train_dset, batch_size=10)

logger = L.pytorch.loggers.TensorBoardLogger(
    ".",
)

### Perform the reconstruction
This cell performs the reconstruction using `train_gpsr_multistep`, which trains `ResNNTransform` in two stages: first only its linear (first-layer) parameters, with the residual network frozen, then the full model (linear + residual network) together. This step will take some time on a CPU but can be greatly accelerated (1-2 orders of magnitude) if using a GPU to do the computation. If you are limited to a CPU I would recommend reducing the number of epochs to reduce computation time.

In [ ]:
litgpsr = train_gpsr_multistep(
    gpsr_model,
    train_loader,
    n_epochs_linear=500,
    n_epochs_full=500,
    logger=logger,
    limit_train_batches=100,
)

### Get the reconstructed beam distribution

In [ ]:
reconstructed_beam = litgpsr.gpsr_model.beam_generator()

### Evaluate model on samples to compare predictions
Here we use the trained GPSR model to make predictions that should agree with the training data. The plot below shows the training data as the colormap and uses contour lines to show the predicted measurements at the 10th, 50th, 95th percentiles.

In [ ]:
test_pred = gpsr_model(test_dset.parameters)[0].detach()
test_pred_dset = QuadScanDataset(test_dset.parameters, (test_pred,), train_dset.screen)

In [ ]:
fig, ax = test_dset.plot_data(overlay_data=test_pred_dset)
fig.set_size_inches(20, 3)

In [ ]:
reconstructed_beam.plot_distribution(dimensions=("x", "px", "y", "py"));

### Compare the linear prediction and reconstruction to the ground truth
`ResNNTransform` first applies a linear transformation (`transformer.linear_forward`) to the base distribution before adding the residual network's nonlinear contribution. Here we compare that linear-only prediction, as well as the final reconstructed beam, to the ground-truth distribution used to generate the training data.

In [ ]:
trained_beam_generator = litgpsr.gpsr_model.beam_generator
linear_beam = to_linear_beam(trained_beam_generator)

# ground truth beam used to generate the training dataset
gt_beam = torch.load(
    "example_data/example_distributions/complex_beam.pt", weights_only=False
)
gt_beam = ParticleBeam(gt_beam.particles, gt_beam.energy)

dims = ("x", "px", "y", "py")

In [ ]:
# compare the linear-only prediction to ground truth
fig, axs = linear_beam.plot_distribution(dimensions=dims)
gt_beam.plot_distribution(dimensions=dims, axs=axs, plot_2d_kws={"style": "contour"})
fig.suptitle("linear prediction vs. ground truth");

In [ ]:
# compare the final reconstruction to ground truth
fig2, axs2 = reconstructed_beam.plot_distribution(dimensions=dims)
gt_beam.plot_distribution(
    dimensions=dims,
    axs=axs2,
    plot_2d_kws={
        "style": "contour",
        "contour_kws": {"levels": [0.05, 0.1, 0.25, 0.5, 0.75, 0.95]},
    },
)
fig2.suptitle("reconstruction vs. ground truth");

## Inspect the strength of the residual network
The `alpha` parameter of the `ResNNTransform` is a learnable scale on the residual correction when fitting the beam distribution. Smaller values suppress the residual branch and make the fitted distribution closer to the linear prediction.

In [ ]:
trained_beam_generator.transformer.alpha